# Desire lines — Toronto's hidden bike highways

If every ride were purely a function of geography, trips between two stations would scale with their joint capacity divided by distance (a gravity model). Reality diverges from gravity — and the pairs that diverge the *most* reveal the invisible infrastructure of Toronto cycling: safe routes, bike lanes, scenic paths, cultural draws.

Pipeline:
1. Compute actual bidirectional trip counts per pair (2025).
2. Fit a simple gravity baseline: expected_trips ~ capacity_A × capacity_B / distance^α.
3. Rank pairs by the ratio `actual / expected`. The top of this list is Toronto's desire lines.

Output: `data/processed/desire_lines_map.html`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import duckdb
import pydeck as pdk

DATA_RAW = Path.cwd().parent / 'data' / 'raw'
DATA_PROC = Path.cwd().parent / 'data' / 'processed'

GLOB = str(DATA_RAW / 'ridership' / '2025' / '*.csv')
con = duckdb.connect()
con.execute(f"CREATE VIEW trips AS SELECT * FROM read_csv_auto('{GLOB}', header=True, ignore_errors=true, union_by_name=true)")

stations = pd.read_parquet(DATA_RAW / 'stations.parquet')
stations['sid'] = pd.to_numeric(stations['station_id'], errors='coerce').astype('Int64')
stations = stations.dropna(subset=['sid']).set_index('sid')[['name', 'lat', 'lon', 'capacity']]
len(stations)

1030

## 1. Bidirectional OD counts

In [2]:
edges = con.execute('''
    SELECT LEAST(Start_Station_Id, End_Station_Id) AS a,
           GREATEST(Start_Station_Id, End_Station_Id) AS b,
           COUNT(*) AS trips
    FROM trips
    WHERE Start_Station_Id IS NOT NULL AND End_Station_Id IS NOT NULL
      AND Start_Station_Id <> End_Station_Id
    GROUP BY 1, 2
''').fetchdf()

# Join coordinates + capacity.
edges = (
    edges
    .join(stations.add_prefix('a_'), on='a', how='inner')
    .join(stations.add_prefix('b_'), on='b', how='inner')
)
print('edges with known endpoints:', len(edges))

edges with known endpoints: 205040


## 2. Haversine distance

In [3]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lat2 = np.radians(lat1), np.radians(lat2)
    dlat = lat2 - lat1
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

edges['dist_km'] = haversine_km(edges['a_lat'], edges['a_lon'], edges['b_lat'], edges['b_lon'])
# Avoid zero (same-location physical cluster) - floor at 50m.
edges['dist_km'] = edges['dist_km'].clip(lower=0.05)
edges[['a', 'b', 'trips', 'a_capacity', 'b_capacity', 'dist_km']].head()

,a,b,trips,a_capacity,b_capacity,dist_km
0,7657,7933,110,20,23,1.664373
1,7030,7377,44,35,36,3.799481
2,7020,7933,138,33,23,1.160034
3,7160,7772,385,19,15,1.637138
4,7409,7600,34,27,20,1.548898


## 3. Fit gravity model

`log(trips) = log(K) + log(cap_a) + log(cap_b) - α * log(dist)` for pairs with trips > 0 and known capacities. We just fit α by least-squares; the coefficients on the capacity terms are fixed at 1 for interpretability.

In [4]:
fit = edges[(edges['a_capacity'] > 0) & (edges['b_capacity'] > 0) & (edges['trips'] > 0)].copy()
fit['log_trips'] = np.log(fit['trips'])
fit['log_cap_prod'] = np.log(fit['a_capacity']) + np.log(fit['b_capacity'])
fit['log_dist'] = np.log(fit['dist_km'])

y = fit['log_trips'] - fit['log_cap_prod']
x = fit['log_dist']
slope, intercept = np.polyfit(x, y, 1)
alpha = -slope
K = np.exp(intercept)
print(f'Fitted gravity: expected_trips = {K:.3f} * cap_a * cap_b / dist_km^{alpha:.2f}')

edges['expected'] = K * edges['a_capacity'].astype(float) * edges['b_capacity'].astype(float) / (edges['dist_km'] ** alpha)
edges['affinity'] = edges['trips'] / edges['expected']

# Filter candidates: real trip volume, real capacity on both ends, and distance > 0.3km
# (pairs <300m apart are essentially the same intersection — physically adjacent docks).
MIN_TRIPS = 300
MIN_DIST = 0.3
candidates = edges[
    (edges['trips'] >= MIN_TRIPS) &
    (edges['a_capacity'] > 0) & (edges['b_capacity'] > 0) &
    (edges['dist_km'] >= MIN_DIST) &
    np.isfinite(edges['affinity'])
].copy()
print(f'{len(candidates)} candidate pairs (>= {MIN_TRIPS} trips, both caps > 0, >= {MIN_DIST} km)')

Fitted gravity: expected_trips = 0.176 * cap_a * cap_b / dist_km^1.59
3348 candidate pairs (>= 300 trips, both caps > 0, >= 0.3 km)


In [5]:
# Top desire lines by affinity.
top = candidates.sort_values('affinity', ascending=False).head(20).copy()
top[['a_name', 'b_name', 'trips', 'dist_km', 'expected', 'affinity']].assign(
    dist_km=lambda d: d['dist_km'].round(2),
    expected=lambda d: d['expected'].round(1),
    affinity=lambda d: d['affinity'].round(1),
)

,a_name,b_name,trips,dist_km,expected,affinity
175933,Palmerston Ave / Vermont Ave,St. Clair Ave W / Castleton Ave,358,5.56,3.3,108.8
176024,Humber Bay Shores Park East,Coronation Park (Martin Goodman Trail),443,5.57,5.0,87.9
52965,Dundas St W / St. Patrick St,Lisgar St / Dundas St W - SMART,340,3.04,4.6,74.3
129,University Ave / Gerrard St W (East Side),Queen St E / Rushbrooke Ave,406,4.60,5.9,69.4
73396,Parkside Dr / Bloor St W - SMART,Tyndall Ave / King St W - SMART,367,3.06,5.7,64.2
77661,Ulster St / Bathurst St,Cumberland St / Bay St - SMART,361,1.90,5.7,63.3
112219,Widmer St / Adelaide St W - SMART,Church St / Shuter St,991,1.47,16.1,61.7
200186,Northern Dancer Blvd / Lake Shore Blvd E,Cherry St / Lake Shore Blvd E,444,4.11,7.3,61.0
73517,Lower Don River Trail and Taylor Creek Trail,Pottery Rd / Lower Don River Trail,588,2.80,9.8,60.1
175796,420 Wellington St W,Shaw St / King St W,1235,1.51,20.7,59.7


In [6]:
# And the opposite: pairs that underperform expectation the most (given enough observations).
# These suggest 'missing infrastructure' — the model says there should be lots of trips, reality disagrees.
bottom = candidates[candidates['expected'] >= 300].sort_values('affinity').head(15).copy()
bottom[['a_name', 'b_name', 'trips', 'dist_km', 'expected', 'affinity']].assign(
    dist_km=lambda d: d['dist_km'].round(2),
    expected=lambda d: d['expected'].round(1),
    affinity=lambda d: d['affinity'].round(3),
)

,a_name,b_name,trips,dist_km,expected,affinity
14721,Bay St / Albert St,Dundas St W / Yonge St,311,0.32,2062.8,0.151
14094,Queen St W / John St,Simcoe St / Pullan Pl,331,0.31,2036.2,0.163
43222,King St E / Berkeley St,Frederick St / King St E,312,0.37,1233.1,0.253
189893,Cherry Beach,Cherry Beach Sports Fields (55 Unwin Ave),308,0.39,1137.9,0.271
27607,Temperance St Station,Bay St / Dundas St W,377,0.58,1253.1,0.301
199653,York St / Queens Quay W,Bay St / Harbour St,364,0.36,1184.2,0.307
102443,Fort York Blvd / Capreol Ct,Fort York Blvd / Bathurst St - SMART,323,0.32,1010.1,0.320
77685,College Park - Gerrard Entrance,Bay St / Dundas St W,348,0.40,1045.1,0.333
108952,Fort York Blvd / Capreol Ct,Bathurst St / Fort York Blvd,357,0.30,1057.0,0.338
52968,Dundas St W / Crawford St,Crawford St / Lobb Ave,323,0.37,929.0,0.348


## 4. Map the desire lines

Top-100 by affinity. Arc colour = log(affinity) on a warm palette (higher = more saturated). Width proportional to trips.

In [7]:
N = 120
top = candidates.sort_values('affinity', ascending=False).head(N).copy()
top = top[np.isfinite(top['affinity']) & (top['affinity'] > 0)].copy()

log_aff = np.log(top['affinity'].astype(float))
q_lo, q_hi = float(log_aff.min()), float(log_aff.max())
def heat(x):
    if not np.isfinite(x):
        return [255, 180, 60, 220]
    t = (x - q_lo) / max(q_hi - q_lo, 1e-9)
    r = 255
    g = int(180 * (1 - t) + 40 * t)
    b = int( 60 * (1 - t) + 30 * t)
    return [r, g, b, 220]
top['color'] = log_aff.map(heat)
top['width'] = np.clip(np.log1p(top['trips'].astype(float)) * 0.9, 1.2, 9.0)
top['pair'] = top['a_name'].astype(str) + ' - ' + top['b_name'].astype(str)
top['aff_r'] = top['affinity'].round(1)
top['trips_k'] = (top['trips'] / 1000).round(2)
top['dist_km_r'] = top['dist_km'].round(2)

arcs = pdk.Layer(
    'ArcLayer',
    data=top[['a_lat', 'a_lon', 'b_lat', 'b_lon', 'color', 'width', 'pair', 'aff_r', 'trips_k', 'dist_km_r']],
    get_source_position=['a_lon', 'a_lat'],
    get_target_position=['b_lon', 'b_lat'],
    get_width='width',
    get_source_color='color',
    get_target_color='color',
    get_height=0.4,
    pickable=True,
    auto_highlight=True,
)

station_points = pdk.Layer(
    'ScatterplotLayer',
    data=stations.reset_index().rename(columns={'sid': 'station_id'})[['name', 'lat', 'lon']],
    get_position=['lon', 'lat'],
    get_radius=28,
    get_fill_color=[180, 180, 180, 180],
    pickable=False,
)

view = pdk.ViewState(latitude=43.660, longitude=-79.395, zoom=12.2, pitch=45, bearing=-15)
deck = pdk.Deck(
    layers=[station_points, arcs],
    initial_view_state=view,
    map_style='light',
    tooltip={'text': '{pair}\n{trips_k}k trips - {dist_km_r} km - affinity {aff_r}x'},
)
out = DATA_PROC / 'desire_lines_map.html'
deck.to_html(str(out), notebook_display=False)
print('Wrote:', out)

Wrote: /Users/tyler/src/open_data_toronto/bikeshare-od-flows/data/processed/desire_lines_map.html


In [8]:
# Short summary — what do the TOP desire lines have in common?
print('Median distance of top-20 desire lines (km):',
      round(candidates.sort_values('affinity', ascending=False).head(20)['dist_km'].median(), 2))
print('Median distance of bottom-20 affinity (same sample):',
      round(candidates.sort_values('affinity').head(20)['dist_km'].median(), 2))
print('Median distance of all candidate pairs:',
      round(candidates['dist_km'].median(), 2))

Median distance of top-20 desire lines (km): 2.92
Median distance of bottom-20 affinity (same sample): 0.36
Median distance of all candidate pairs: 1.13
